<a href="https://colab.research.google.com/github/Alvarojm13/tfm-cleveland/blob/main/02_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFM · Notebook 02 — Partición, preprocesado y entrenamiento

**Objetivo (Fase 4 del plan):**

1. Dividir los datos en entrenamiento (80 %) y test (20 %), **una sola vez**.
2. Construir el `Pipeline`: imputación, escalado y one-hot, ajustados solo con el entrenamiento.
3. Entrenar con validación cruzada los modelos acordados:
   - Comparador reentrenado: logística con edad, sexo y tipo de dolor.
   - Regresión logística L2 y Random Forest, cada uno con las variables de **consulta** y las del modelo **completo**.
4. Preparar la tabla de Diamond y Forrester (no se entrena).
5. Fijar el umbral de sensibilidad ≥ 90 % **con el entrenamiento**.
6. Guardar todo en un único archivo para el notebook 03.

> ⚠️ **En este notebook el conjunto de test no se mira.** Solo se separa y se guarda.
> Toda decisión (hiperparámetros, umbrales) se toma con el 80 % de entrenamiento.

**En Colab:** `Entorno de ejecución → Ejecutar todas` y sube `datos_cleveland_limpios.csv` cuando lo pida.

## 1. Librerías

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import joblib                                   # para guardar los modelos entrenados
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEMILLA = 42
print("scikit-learn", sklearn.__version__)   # anótalo en métodos (apartado de software)

scikit-learn 1.6.1


## 2. Carga de los datos limpios del notebook 01

In [ ]:
try:
    from google.colab import files
    if not glob.glob("datos_cleveland_limpios*.csv"):
        print("Sube datos_cleveland_limpios.csv:")
        files.upload()
except ImportError:
    pass

# Acepta el nombre aunque el navegador le haya añadido " (1)"
fichero = sorted(glob.glob("datos_cleveland_limpios*.csv"))[0]
df = pd.read_csv(fichero)
print(fichero, df.shape, "| eventos:", df["enfermedad"].sum())
print("Perdidos (se imputarán dentro del Pipeline):", df[["ca", "thal"]].isna().sum().to_dict())

Sube datos_cleveland_limpios.csv:


Saving datos_cleveland_limpios (1).csv to datos_cleveland_limpios (1).csv
datos_cleveland_limpios (1).csv (303, 16) | eventos: 139
Perdidos (se imputarán dentro del Pipeline): {'ca': 4, 'thal': 2}


## 3. Conjuntos de variables

Los tres niveles de información están **anidados**: cada uno añade variables al anterior.

In [ ]:
VARS_DF       = ["age", "sex", "cp"]                                            # Diamond-Forrester
VARS_CONSULTA = VARS_DF + ["trestbps", "chol", "fbs", "restecg_anormal"]          # 7 variables
VARS_COMPLETO = VARS_CONSULTA + ["thalach", "exang", "oldpeak", "slope", "ca", "thal"]  # 13 variables

# Tipo de cada variable -> cómo se preprocesa
CONTINUAS   = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]  # ca: 0-3 vasos, numérica
NOMINALES   = ["cp", "slope", "thal"]                                  # -> one-hot
BINARIAS    = ["sex", "fbs", "restecg_anormal", "exang"]               # ya son 0/1

## 4. Partición 80/20 estratificada (una sola vez)

`stratify` mantiene la misma proporción de enfermos en ambos conjuntos.
Se guardan los índices para que el notebook 03 use exactamente la misma partición.

In [ ]:
X = df.drop(columns=["num", "enfermedad", "restecg"])
y = df["enfermedad"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA)

print(f"Entrenamiento: {len(X_train)} pacientes, {y_train.sum()} eventos ({y_train.mean():.1%})")
print(f"Test:          {len(X_test)} pacientes, {y_test.sum()} eventos ({y_test.mean():.1%})")
print("Perdidos en entrenamiento:", X_train[["ca", "thal"]].isna().sum().sum(),
      "| en test:", X_test[["ca", "thal"]].isna().sum().sum())

Entrenamiento: 242 pacientes, 111 eventos (45.9%)
Test:          61 pacientes, 28 eventos (45.9%)
Perdidos en entrenamiento: 2 | en test: 4


## 5. El preprocesado dentro del Pipeline

Para cada grupo de variables:

| Grupo | Imputación (si falta un dato) | Transformación |
|---|---|---|
| Continuas | moda (`ca`) o mediana | estandarización (media 0, DE 1) |
| Nominales | moda (`thal`) | one-hot (una columna 0/1 por categoría) |
| Binarias | — | ninguna |

Como todo va dentro del `Pipeline`, la moda, las medias y las DE se calculan **solo con
los datos de entrenamiento de cada pliegue**. Esto es lo que evita la fuga de información.

In [ ]:
def preprocesador(variables):
    # Construye el preprocesado solo para las variables que use cada modelo.
    cont = [v for v in CONTINUAS if v in variables]
    nom  = [v for v in NOMINALES if v in variables]
    bina = [v for v in BINARIAS  if v in variables]
    pasos = []
    if cont:
        pasos.append(("continuas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),  # solo afecta a ca
            ("escalar", StandardScaler())]), cont))
    if nom:
        pasos.append(("nominales", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),  # solo afecta a thal
            ("onehot", OneHotEncoder(handle_unknown="ignore"))]), nom))
    if bina:
        pasos.append(("binarias", "passthrough", bina))
    return ColumnTransformer(pasos)

## 6. Validación cruzada y rejillas de hiperparámetros

- 5 pliegues estratificados, solo sobre el 80 % de entrenamiento.
- Métrica para elegir hiperparámetros: AUC-ROC.
- Rejillas pequeñas a propósito: con 242 pacientes, rejillas grandes sobreajustan la propia búsqueda.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

REJILLA_LR = {"modelo__C": [0.001, 0.01, 0.1, 1, 10, 100]}   # C pequeño = más regularización
REJILLA_RF = {"modelo__n_estimators": [300],
              "modelo__max_depth": [3, 5, None],
              "modelo__min_samples_leaf": [1, 5, 10],
              "modelo__max_features": ["sqrt"]}

def entrenar(nombre, variables, modelo, rejilla):
    pipe = Pipeline([("prep", preprocesador(variables)), ("modelo", modelo)])
    if rejilla:
        busqueda = GridSearchCV(pipe, rejilla, cv=CV, scoring="roc_auc", n_jobs=-1)
        busqueda.fit(X_train[variables], y_train)
        mejor, params, auc_cv = busqueda.best_estimator_, busqueda.best_params_, busqueda.best_score_
        sd_cv = busqueda.cv_results_["std_test_score"][busqueda.best_index_]
    else:  # modelo sin hiperparámetros: solo se estima su AUC en validación cruzada
        aucs = [roc_auc_score(y_train.iloc[te],
                              pipe.fit(X_train[variables].iloc[tr], y_train.iloc[tr])
                                  .predict_proba(X_train[variables].iloc[te])[:, 1])
                for tr, te in CV.split(X_train, y_train)]
        mejor, params, auc_cv, sd_cv = pipe.fit(X_train[variables], y_train), {}, np.mean(aucs), np.std(aucs)
    print(f"{nombre:<28} AUC (CV entrenamiento) = {auc_cv:.3f} ± {sd_cv:.3f}  {params}")
    return {"modelo": mejor, "variables": variables, "params": params,
            "auc_cv": auc_cv, "sd_cv": sd_cv}

## 7. Entrenamiento de los cinco modelos

In [ ]:
modelos = {}
modelos["Logística DF reentrenada"] = entrenar(
    "Logística DF reentrenada", VARS_DF, LogisticRegression(C=1e6, max_iter=1000), None)  # C enorme = sin penalización
modelos["Logística consulta"] = entrenar(
    "Logística consulta", VARS_CONSULTA, LogisticRegression(max_iter=1000), REJILLA_LR)
modelos["Logística completa"] = entrenar(
    "Logística completa", VARS_COMPLETO, LogisticRegression(max_iter=1000), REJILLA_LR)
modelos["Random Forest consulta"] = entrenar(
    "Random Forest consulta", VARS_CONSULTA, RandomForestClassifier(random_state=SEMILLA), REJILLA_RF)
modelos["Random Forest completo"] = entrenar(
    "Random Forest completo", VARS_COMPLETO, RandomForestClassifier(random_state=SEMILLA), REJILLA_RF)

Logística DF reentrenada     AUC (CV entrenamiento) = 0.830 ± 0.040  {}
Logística consulta           AUC (CV entrenamiento) = 0.828 ± 0.044  {'modelo__C': 10}
Logística completa           AUC (CV entrenamiento) = 0.911 ± 0.019  {'modelo__C': 0.1}
Random Forest consulta       AUC (CV entrenamiento) = 0.825 ± 0.024  {'modelo__max_depth': 3, 'modelo__max_features': 'sqrt', 'modelo__min_samples_leaf': 1, 'modelo__n_estimators': 300}
Random Forest completo       AUC (CV entrenamiento) = 0.900 ± 0.024  {'modelo__max_depth': 5, 'modelo__max_features': 'sqrt', 'modelo__min_samples_leaf': 10, 'modelo__n_estimators': 300}


**Cómo leer esto:** es el AUC medio en los 5 pliegues de entrenamiento. Sirve para elegir
hiperparámetros y para detectar problemas, **no** es el resultado final del TFM: ese sale del
test en el notebook 03. Una diferencia pequeña aquí no significa que un modelo sea mejor.

## 8. Tabla de Diamond y Forrester (1979)

No se entrena: es una tabla de consulta. Valores en % transcritos del artículo original
(Tabla 3 para sintomáticos, Tabla 2 —prevalencia en autopsias— para asintomáticos).
Edades fuera de 30–69 años: se asigna la década más cercana.

In [ ]:
# filas: décadas 30-39, 40-49, 50-59, 60-69 | claves: (tipo de dolor cp, sexo)  sexo: 1 = hombre, 0 = mujer
TABLA_DF = {
    (3, 1): [5.2, 14.1, 21.5, 28.1],  (3, 0): [0.8, 2.8, 8.4, 18.6],    # dolor no anginoso
    (2, 1): [21.8, 46.1, 58.9, 67.1], (2, 0): [4.2, 13.3, 32.4, 54.4],  # angina atípica
    (1, 1): [69.7, 87.3, 92.0, 94.3], (1, 0): [25.8, 55.2, 79.4, 90.6], # angina típica
    (4, 1): [1.9, 5.5, 9.7, 12.3],    (4, 0): [0.3, 1.0, 3.2, 7.5],     # asintomático (Tabla 2)
}

def prob_diamond_forrester(datos):
    decada = ((datos["age"] // 10) - 3).clip(0, 3).astype(int)   # 29 -> 30-39; >=70 -> 60-69
    return np.array([TABLA_DF[(int(cp), int(sx))][d] / 100
                     for cp, sx, d in zip(datos["cp"], datos["sex"], decada)])

# Comprobación con dos pacientes de ejemplo
ejemplo = pd.DataFrame({"age": [55, 72], "sex": [1, 0], "cp": [1, 4]})
print(prob_diamond_forrester(ejemplo))   # esperado: 0.92 y 0.075

[0.92  0.075]


## 9. Umbral de *rule-out* (sensibilidad ≥ 90 %) fijado con el entrenamiento

Para cada modelo se obtienen predicciones **fuera de pliegue** en el entrenamiento
(cada paciente es predicho por un modelo que no lo vio) y se busca el umbral más alto
que todavía alcanza una sensibilidad ≥ 90 %. El umbral queda fijo; en el test solo se aplica.

In [ ]:
def umbral_sensibilidad(y_real, prob, objetivo=0.90):
    candidatos = np.sort(np.unique(prob))[::-1]          # de mayor a menor
    for u in candidatos:
        if ((prob >= u) & (y_real == 1)).sum() / (y_real == 1).sum() >= objetivo:
            return float(u)
    return float(candidatos[-1])

umbrales = {}
for nombre, m in modelos.items():
    prob_oof = cross_val_predict(m["modelo"], X_train[m["variables"]], y_train,
                                 cv=CV, method="predict_proba")[:, 1]
    umbrales[nombre] = umbral_sensibilidad(y_train.values, prob_oof)
umbrales["Diamond-Forrester (tabla)"] = umbral_sensibilidad(
    y_train.values, prob_diamond_forrester(X_train))

pd.Series(umbrales, name="Umbral sensibilidad ≥ 90 %").round(3)

,Umbral sensibilidad ≥ 90 %
Logística DF reentrenada,0.265
Logística consulta,0.248
Logística completa,0.300
Random Forest consulta,0.325
Random Forest completo,0.330
Diamond-Forrester (tabla),0.055


## 10. Resumen del entrenamiento y guardado

In [ ]:
resumen = pd.DataFrame({
    "Variables": {k: len(v["variables"]) for k, v in modelos.items()},
    "AUC CV (media)": {k: round(v["auc_cv"], 3) for k, v in modelos.items()},
    "AUC CV (DE)": {k: round(v["sd_cv"], 3) for k, v in modelos.items()},
    "Hiperparámetros": {k: v["params"] for k, v in modelos.items()},
})
resumen.to_csv("resumen_entrenamiento.csv")
display(resumen)

paquete = {"modelos": modelos, "umbrales": umbrales, "tabla_df": TABLA_DF,
           "indices_train": X_train.index.tolist(), "indices_test": X_test.index.tolist(),
           "semilla": SEMILLA, "sklearn": sklearn.__version__}
joblib.dump(paquete, "modelos_tfm.joblib")
print("Guardado modelos_tfm.joblib")
try:
    from google.colab import files
    files.download("modelos_tfm.joblib")
    files.download("resumen_entrenamiento.csv")
except ImportError:
    pass

,Variables,AUC CV (media),AUC CV (DE),Hiperparámetros
Logística DF reentrenada,3,0.830,0.040,{}
Logística consulta,7,0.828,0.044,{'modelo__C': 10}
Logística completa,13,0.911,0.019,{'modelo__C': 0.1}
Random Forest consulta,7,0.825,0.024,"{'modelo__max_depth': 3, 'modelo__max_features..."
Random Forest completo,13,0.900,0.024,"{'modelo__max_depth': 5, 'modelo__max_features..."


Guardado modelos_tfm.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Qué llevar a la memoria

- **Métodos:** partición 80/20 estratificada (semilla 42), preprocesado dentro del `Pipeline`
  (imputación por moda de `ca` y `thal`, estandarización, one-hot), validación cruzada de
  5 pliegues con AUC como métrica, rejillas de hiperparámetros (tabla en anexo), regla para
  edades fuera de la tabla de Diamond y Forrester, y el criterio del umbral de *rule-out*.
- **Anexo:** tabla `resumen_entrenamiento.csv` con los hiperparámetros elegidos.
- **No** pongas aún resultados de rendimiento: los resultados del TFM salen del test (notebook 03).